# 17 — Inventory Optimisation
## SunnyBest Retail Forecasting System

> **Purpose:** Use actual demand data and forecasts to compute the optimal reorder point, safety stock and order quantity for every store × product combination.

### The problem
Right now SunnyBest restocks on a fixed schedule (Large = every 3 days, Medium = 5, Small = 7) with no regard to how variable demand actually is. A product with very predictable demand needs much less safety stock than one that swings wildly. This notebook replaces intuition with maths.

### Key formulas

| Term | Formula | What it means |
|------|---------|---------------|
| **Safety Stock** | Z × σ_demand × √lead_time | Buffer against demand variability |
| **Reorder Point** | avg_demand × lead_time + safety_stock | Order when stock hits this level |
| **EOQ** | √(2 × D × S / H) | Optimal order quantity balancing order cost vs holding cost |
| **Service Level** | Z = 1.65 → 95% | How often you want to avoid a stockout |

Where:  
- **D** = annual demand (units/year)  
- **S** = order cost per order (₦)  
- **H** = holding cost per unit per year (₦)  
- **Z** = service level z-score  
- **σ** = standard deviation of daily demand  
- **lead_time** = days between placing and receiving an order

---
## 0. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from scipy import stats

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

plt.rcParams["figure.figsize"]    = (14, 5)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False
PALETTE = ["#4C72B0","#C44E52","#55A868","#DD8452","#8172B2","#64B5CD","#CCB974"]

# ── Tunable parameters ────────────────────────────────────
SERVICE_LEVEL    = 0.95   # 95% — stockout happens only 5% of the time
Z_SCORE          = stats.norm.ppf(SERVICE_LEVEL)  # = 1.645

ORDER_COST       = 5000   # ₦ cost per order placed (delivery, admin)
HOLDING_COST_PCT = 0.20   # 20% of unit cost per year (storage, insurance, obsolescence)

# Lead times by store size (days between order and delivery)
LEAD_TIMES = {"Large": 2, "Medium": 3, "Small": 5}

# DB connection
host     = "aws-1-eu-central-1.pooler.supabase.com"
port     = 5432
database = "postgres"
user     = "postgres.ogkdfmkybqtrsglcizzt"
password = quote_plus(os.getenv("SUPABASE_DB_PASSWORD", "Bonabosssfs01"))

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}",
    pool_pre_ping=True, pool_recycle=300
)

print(f"Service level : {SERVICE_LEVEL:.0%}  (Z = {Z_SCORE:.3f})")
print(f"Order cost    : ₦{ORDER_COST:,}")
print(f"Holding cost  : {HOLDING_COST_PCT:.0%} of unit cost per year")
print(f"Lead times    : {LEAD_TIMES}")

---
## 1. Load Demand Data

> We need daily demand per store × product to compute variability.  
> Using last 12 months for a stable, current estimate.

In [ ]:
df = pd.read_sql("""
    SELECT
        s.date,
        s.store_id,
        st.store_name,
        st.store_size,
        s.product_id,
        p.product_name,
        p.category,
        p.cost_price,
        p.regular_price,
        s.units_sold,
        s.stockout_occurred,
        s.starting_inventory
    FROM core.fact_sales s
    LEFT JOIN core.dim_stores   st ON s.store_id   = st.store_id
    LEFT JOIN core.dim_products  p ON s.product_id  = p.product_id
    WHERE s.date >= NOW() - INTERVAL '12 months'
    ORDER BY s.date ASC
""", engine)

df["date"] = pd.to_datetime(df["date"])

print(f"Rows      : {len(df):,}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Stores    : {df['store_id'].nunique()}")
print(f"Products  : {df['product_id'].nunique()}")

---
## 2. Compute Demand Statistics per Store × Product

> For each store-product combination, we need:  
> - Average daily demand  
> - Standard deviation (variability)  
> - Stockout rate (how often it runs out)  
> - Current average inventory level

In [ ]:
demand_stats = (df.groupby(["store_id","store_name","store_size","product_id","product_name","category","cost_price","regular_price"])
                  .agg(
                      avg_daily_demand  =("units_sold",         "mean"),
                      std_daily_demand  =("units_sold",         "std"),
                      total_units_sold  =("units_sold",         "sum"),
                      days_observed     =("units_sold",         "count"),
                      stockout_rate     =("stockout_occurred",  "mean"),
                      avg_inventory     =("starting_inventory", "mean"),
                  )
                  .reset_index())

demand_stats["std_daily_demand"] = demand_stats["std_daily_demand"].fillna(0)
demand_stats["annual_demand"]    = demand_stats["avg_daily_demand"] * 365
demand_stats["holding_cost"]     = demand_stats["cost_price"] * HOLDING_COST_PCT
demand_stats["lead_time"]        = demand_stats["store_size"].map(LEAD_TIMES)

print(f"Store × product combinations: {len(demand_stats):,}")
print(f"\nDemand stats summary:")
display(demand_stats[["store_name","product_name","category","avg_daily_demand",
                       "std_daily_demand","stockout_rate","avg_inventory"]].head(15))

---
## 3. Safety Stock

> **Safety Stock = Z × σ × √lead_time**  
> Products with high demand variability need more buffer.  
> Products with long lead times need more buffer.

In [ ]:
demand_stats["safety_stock"] = (
    Z_SCORE
    * demand_stats["std_daily_demand"]
    * np.sqrt(demand_stats["lead_time"])
).round(1)

# ── Safety stock by category ──────────────────────────────
ss_cat = (demand_stats.groupby("category")
            .agg(avg_safety_stock=("safety_stock","mean"),
                 max_safety_stock=("safety_stock","max"))
            .reset_index()
            .sort_values("avg_safety_stock", ascending=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(ss_cat["category"], ss_cat["avg_safety_stock"], color=PALETTE[:len(ss_cat)])
ax.set_title("Average Safety Stock by Category\n(units needed as buffer at 95% service level)")
ax.set_ylabel("Safety Stock (units)")
ax.tick_params(axis="x", rotation=30)
for i, row in ss_cat.reset_index().iterrows():
    ax.text(i, row["avg_safety_stock"] + 0.1, f"{row['avg_safety_stock']:.1f}",
            ha="center", fontsize=8)
plt.tight_layout()
plt.show()

display(ss_cat)

---
## 4. Reorder Point

> **Reorder Point = avg_demand × lead_time + safety_stock**  
> When stock falls to this level, place an order immediately.  
> The stock will arrive exactly when you're about to run out.

In [ ]:
demand_stats["reorder_point"] = (
    demand_stats["avg_daily_demand"] * demand_stats["lead_time"]
    + demand_stats["safety_stock"]
).round(1)

# Flag products currently below reorder point
demand_stats["below_reorder"] = demand_stats["avg_inventory"] < demand_stats["reorder_point"]

below = demand_stats[demand_stats["below_reorder"]].sort_values("stockout_rate", ascending=False)

print(f"Products currently below their reorder point: {len(below):,}")
print(f"Total store-product combinations tracked    : {len(demand_stats):,}")
print(f"Pct below reorder point                     : {len(below)/len(demand_stats):.1%}")

print("\nTop 20 — highest stockout risk (below reorder point, sorted by stockout rate):")
display(below[["store_name","product_name","category","avg_inventory",
               "reorder_point","safety_stock","stockout_rate"]].head(20).reset_index(drop=True))

---
## 5. Economic Order Quantity (EOQ)

> **EOQ = √(2 × D × S / H)**  
> The order quantity that minimises total cost (ordering cost + holding cost).  
> Order too little → too many orders, high admin cost.  
> Order too much → too much stock sitting, high holding cost.  
> EOQ is the sweet spot.

In [ ]:
demand_stats["eoq"] = np.where(
    demand_stats["holding_cost"] > 0,
    np.sqrt(
        2 * demand_stats["annual_demand"] * ORDER_COST / demand_stats["holding_cost"]
    ).round(0),
    np.nan
)

# EOQ by category
eoq_cat = (demand_stats.groupby("category")
             .agg(avg_eoq   =("eoq",   "mean"),
                  avg_demand=("avg_daily_demand","mean"))
             .reset_index()
             .sort_values("avg_eoq", ascending=False))
eoq_cat["avg_eoq"] = eoq_cat["avg_eoq"].round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(eoq_cat["category"], eoq_cat["avg_eoq"], color=PALETTE[:len(eoq_cat)])
axes[0].set_title("Average EOQ by Category\n(optimal units per order)")
axes[0].set_ylabel("EOQ (units)")
axes[0].tick_params(axis="x", rotation=30)

# EOQ vs Daily demand scatter
for i, (cat, grp) in enumerate(demand_stats.groupby("category")):
    axes[1].scatter(grp["avg_daily_demand"], grp["eoq"],
                    label=cat, alpha=0.6, s=30, color=PALETTE[i % len(PALETTE)])
axes[1].set_xlabel("Avg Daily Demand")
axes[1].set_ylabel("EOQ (units)")
axes[1].set_title("EOQ vs Daily Demand")
axes[1].legend(fontsize=7, title="Category")

plt.tight_layout()
plt.show()

display(eoq_cat)

---
## 6. Full Inventory Policy Table

> The complete action table — one row per store × product.  
> This is what you hand to procurement: when to order and how much.

In [ ]:
policy = demand_stats[[
    "store_name", "store_size", "product_name", "category",
    "avg_daily_demand", "std_daily_demand",
    "lead_time", "safety_stock", "reorder_point",
    "eoq", "avg_inventory", "stockout_rate", "below_reorder"
]].copy()

policy["avg_daily_demand"] = policy["avg_daily_demand"].round(2)
policy["std_daily_demand"] = policy["std_daily_demand"].round(2)
policy["stockout_rate"]    = (policy["stockout_rate"] * 100).round(1)
policy["avg_inventory"]    = policy["avg_inventory"].round(1)
policy["action"]           = policy["below_reorder"].map({True: "🔴 ORDER NOW", False: "✅ OK"})

policy = policy.sort_values(["below_reorder","stockout_rate"], ascending=[False, False])

print(f"Full inventory policy — {len(policy):,} store × product combinations")
display(policy.drop(columns="below_reorder").head(30).reset_index(drop=True))

---
## 7. Inventory Policy Summary by Store

> Which stores need the most urgent attention?

In [ ]:
store_summary = (demand_stats.groupby(["store_name","store_size"])
                   .agg(
                       products_below_reorder=("below_reorder","sum"),
                       total_products        =("product_id",   "count"),
                       avg_stockout_rate     =("stockout_rate","mean"),
                       avg_safety_stock      =("safety_stock", "mean"),
                   )
                   .reset_index())

store_summary["pct_below_reorder"]  = (store_summary["products_below_reorder"] / store_summary["total_products"] * 100).round(1)
store_summary["avg_stockout_rate"]  = (store_summary["avg_stockout_rate"] * 100).round(2)
store_summary["avg_safety_stock"]   = store_summary["avg_safety_stock"].round(1)
store_summary = store_summary.sort_values("pct_below_reorder", ascending=False)

display(store_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(store_summary["store_name"], store_summary["pct_below_reorder"],
            color=[PALETTE[1] if v > 20 else PALETTE[0] for v in store_summary["pct_below_reorder"]])
axes[0].set_title("% of Products Below Reorder Point by Store\n(red = >20% need restocking)")
axes[0].set_ylabel("% below reorder point")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(store_summary["store_name"], store_summary["avg_safety_stock"],
            color=PALETTE[:len(store_summary)])
axes[1].set_title("Average Safety Stock Required by Store")
axes[1].set_ylabel("Safety Stock (units)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

---
## 8. Compare Current Inventory vs Optimal Policy

> Visual gap analysis — how far is actual inventory from where it should be?

In [ ]:
cat_policy = (demand_stats.groupby("category")
                .agg(
                    avg_current_inventory=("avg_inventory",  "mean"),
                    avg_reorder_point    =("reorder_point",  "mean"),
                    avg_safety_stock     =("safety_stock",   "mean"),
                )
                .reset_index()
                .sort_values("avg_reorder_point", ascending=False))

x   = np.arange(len(cat_policy))
w   = 0.3

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w,   cat_policy["avg_current_inventory"], w, label="Current avg inventory", color=PALETTE[0])
ax.bar(x,       cat_policy["avg_reorder_point"],     w, label="Reorder point",          color=PALETTE[1])
ax.bar(x + w,   cat_policy["avg_safety_stock"],      w, label="Safety stock",           color=PALETTE[2])

ax.set_xticks(x)
ax.set_xticklabels(cat_policy["category"], rotation=30, ha="right")
ax.set_ylabel("Units")
ax.set_title("Current Inventory vs Optimal Policy by Category")
ax.legend()
plt.tight_layout()
plt.show()

print("\nCategories where current inventory is BELOW reorder point on average:")
below_cat = cat_policy[cat_policy["avg_current_inventory"] < cat_policy["avg_reorder_point"]]
if len(below_cat):
    display(below_cat.reset_index(drop=True))
else:
    print("All categories are above reorder point on average.")

---
## Insights

**What this analysis gives SunnyBest that they don't have today:**

The current system restocks on a fixed schedule regardless of demand variability. This notebook replaces that with a **data-driven policy** for every store × product combination.

**Key findings to look for:**

1. **Products below reorder point** → these need orders placed today, not on the next scheduled delivery

2. **High safety stock categories** → categories with high demand variability (Mobile Phones, Accessories) need proportionally more buffer than stable categories (Refrigerators)

3. **Small stores vs Large stores** → small stores have longer lead times (5 days vs 2) so they need to reorder earlier despite lower volumes — this is counterintuitive and why the fixed schedule fails them

4. **EOQ tells you order quantity, reorder point tells you order timing** — these are two separate decisions. Most businesses only think about one

**How to adjust the parameters:**
- Raise `SERVICE_LEVEL` to 0.99 → more safety stock, fewer stockouts, higher holding cost
- Lower `ORDER_COST` → smaller, more frequent orders become optimal (EOQ decreases)
- Change `LEAD_TIMES` → if a new supplier is faster, safety stock requirements drop immediately